# 🧠 Optimizers
**Course:** Deep Learning  
**Lecture:** 4 — Optimizers and Regularization  
**Part:** a — Optimizers

---

> **Where we are:** Now that we know how to construct networks, we need to know how to efficiently update their weights to minimize the loss function.

**What you'll learn:**
- Standard gradient descent and its limitations
- Momentum and Nesterov accelerated gradient
- Adaptive learning rate methods like RMSProp and Adam
- How to use and compare them in TensorFlow

**Exam relevance:** ⭐⭐⭐ (Very important! You will likely be asked to explain Adam or compare it to SGD with momentum.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Optimization Algorithms
When training a neural network, we use gradient descent to update weights $w$. 

- **SGD (Vanilla):** $w \leftarrow w - \eta \nabla L$. Slow, can get stuck in local minima or saddle points.
- **SGD + Momentum:** Accumulates a velocity vector to build up speed in consistent directions.
  $v \leftarrow \beta v + \nabla L$
  $w \leftarrow w - \eta v$
- **RMSProp:** Divides learning rate by a moving average of squared gradients to adapt per-weight.
- **Adam:** Combines Momentum (first moment $m$) and RMSProp (second moment $v$). The gold standard.
  $m \leftarrow \beta_1 m + (1-\beta_1) \nabla L$
  $v \leftarrow \beta_2 v + (1-\beta_2) \nabla L^2$
  Bias correction is applied in the first few steps.
- **AdamW:** Adam with decoupled weight decay.


## 🔀 Your Options
At this step, you can choose between several optimizers. Here is a comparison:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| **SGD** | Simple, theoretically sound | Very slow, needs careful tuning | Rarely used vanilla |
| **SGD+Momentum** | Much faster convergence | Hyperparameter $\beta$ needs tuning | Often for vision tasks (e.g., ResNets) |
| **RMSProp** | Adapts LR per weight | Can stall if LR drops too fast | RNNs, sequential data |
| **Adam** | Fast, default choice, adaptive | Might overfit compared to SGD | Almost everywhere as a default |
| **AdamW** | Better generalization than Adam | Slightly more tuning for weight decay | Modern transformers, heavy vision models |

### Part 1: Visualizing Optimizers on a 2D Loss Surface (Rosenbrock)

In [ ]:
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])


In [ ]:
# Implement standard SGD update rule
def sgd_update(w, grad, lr):
    w_new = w - lr * grad  # gradient descent step
    return w_new

# Implement SGD + Momentum
def momentum_update(w, grad, lr, v, beta=0.9):
    v_new = beta * v + grad           # accumulate velocity
    w_new = w - lr * v_new            # update weights using velocity
    return w_new, v_new

# Implement Adam with bias correction
def adam_update(w, grad, lr, m, v, t, beta1=0.9, beta2=0.999, eps=1e-8):
    m_new = beta1 * m + (1 - beta1) * grad          # first moment (mean)
    v_new = beta2 * v + (1 - beta2) * grad**2        # second moment (variance)
    m_hat = m_new / (1 - beta1**t)                  # bias correction step 1
    v_hat = v_new / (1 - beta2**t)                  # bias correction step 2
    w_new = w - lr * m_hat / (np.sqrt(v_hat) + eps) # parameter update
    return w_new, m_new, v_new

# Quick test on a simple quadratic f(w) = w^2, gradient = 2w
w = np.array([3.0])
losses_sgd, losses_mom, losses_adam = [], [], []
w_sgd, w_mom, w_adam = w.copy(), w.copy(), w.copy()
v_mom = np.zeros_like(w)
m_a, v_a = np.zeros_like(w), np.zeros_like(w)

for t in range(1, 51):
    g = 2 * w_sgd;  w_sgd = sgd_update(w_sgd, g, lr=0.05);                        losses_sgd.append(float(w_sgd**2))
    g = 2 * w_mom;  w_mom, v_mom = momentum_update(w_mom, g, 0.05, v_mom);        losses_mom.append(float(w_mom**2))
    g = 2 * w_adam; w_adam, m_a, v_a = adam_update(w_adam, g, 0.1, m_a, v_a, t); losses_adam.append(float(w_adam**2))

plt.figure(figsize=(8, 4))
plt.plot(losses_sgd,  label='SGD')
plt.plot(losses_mom,  label='SGD+Momentum')
plt.plot(losses_adam, label='Adam')
plt.xlabel('Step'); plt.ylabel('f(w) = w^2')
plt.title('Optimizer comparison on a quadratic loss surface')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
### ✅ SOLUTION ###
def sgd_update(w, grad, lr):
    return w - lr * grad

def momentum_update(w, grad, lr, v, beta):
    v_new = beta * v + grad
    w_new = w - lr * v_new
    return w_new, v_new

def adam_update(w, grad, lr, m, v, t, beta1=0.9, beta2=0.999, eps=1e-8):
    m_new = beta1 * m + (1 - beta1) * grad
    v_new = beta2 * v + (1 - beta2) * (grad ** 2)
    
    # Bias correction
    m_hat = m_new / (1 - beta1 ** t)
    v_hat = v_new / (1 - beta2 ** t)
    
    w_new = w - lr * m_hat / (np.sqrt(v_hat) + eps)
    return w_new, m_new, v_new


In [ ]:
### 🔬 Comparison Cell: Trajectories
np.random.seed(42)
start_w = np.array([-1.0, -1.0])
iterations = 1000

def run_optimizer(opt_name, start_w, iterations):
    w = start_w.copy()
    path = [w.copy()]
    
    # States
    v_mom = np.zeros_like(w)
    m_adam, v_adam = np.zeros_like(w), np.zeros_like(w)
    
    for t in range(1, iterations + 1):
        g = rosenbrock_grad(w[0], w[1])
        
        if opt_name == 'SGD':
            w = sgd_update(w, g, lr=0.001)
        elif opt_name == 'Momentum':
            w, v_mom = momentum_update(w, g, lr=0.001, v=v_mom, beta=0.9)
        elif opt_name == 'Adam':
            w, m_adam, v_adam = adam_update(w, g, lr=0.1, m=m_adam, v=v_adam, t=t)
            
        path.append(w.copy())
    return np.array(path)

path_sgd = run_optimizer('SGD', start_w, iterations)
path_mom = run_optimizer('Momentum', start_w, iterations)
path_adam = run_optimizer('Adam', start_w, iterations)

# Plot
X, Y = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-1, 3, 100))
Z = rosenbrock(X, Y)

plt.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='jet')
plt.plot(path_sgd[:,0], path_sgd[:,1], label='SGD', alpha=0.8)
plt.plot(path_mom[:,0], path_mom[:,1], label='Momentum', alpha=0.8)
plt.plot(path_adam[:,0], path_adam[:,1], label='Adam', alpha=0.8)
plt.plot(1, 1, 'r*', markersize=15, label='Global Min')
plt.legend()
plt.title('Optimizer Trajectories on Rosenbrock Function')
plt.show()


### Part 2: Training a MLP on MNIST

In [ ]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

def build_model():
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])


In [ ]:
optimizers_to_test = {
    'SGD':      tf.keras.optimizers.SGD(learning_rate=0.01),
    'Momentum': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Nesterov': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    'RMSProp':  tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam':     tf.keras.optimizers.Adam(learning_rate=0.001),
    'AdamW':    tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4),
}
print('Optimizers ready:', list(optimizers_to_test.keys()))

In [ ]:
### ✅ SOLUTION ###
optimizers_to_test = {
    'SGD': tf.keras.optimizers.SGD(learning_rate=0.01),
    'Momentum': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSProp': tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam': tf.keras.optimizers.Adam(learning_rate=0.001),
    'AdamW': tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4)
}

histories = {}
for name, opt in optimizers_to_test.items():
    print(f"Training with {name}...")
    tf.random.set_seed(42)
    model = build_model()
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test), verbose=0)
    histories[name] = hist


In [ ]:
### 🔬 Comparison Cell: Loss & Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for name, hist in histories.items():
    ax1.plot(hist.history['loss'], label=name)
    ax2.plot(hist.history['val_accuracy'], label=name)

ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.show()


## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the Adam update rule. What are $\beta_1$ and $\beta_2$?

<details><summary>💡 Answer</summary>

$m_t = \beta_1 m_{t-1} + (1-\beta_1) \nabla L$
$v_t = \beta_2 v_{t-1} + (1-\beta_2) \nabla L^2$
$\hat{m}_t = m_t / (1 - \beta_1^t)$ (bias correction)
$\hat{v}_t = v_t / (1 - \beta_2^t)$ (bias correction)
$w_{t} = w_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

$\beta_1$ controls the momentum (decay of first moment). $\beta_2$ controls RMSProp part (decay of second moment/variance).

</details>

### Question 2 (Conceptual)
What is the difference between SGD with momentum and Adam?

<details><summary>💡 Answer</summary>

SGD with momentum uses a single global learning rate and accumulates past gradients. Adam additionally maintains a moving average of squared gradients (second moment) to adjust the learning rate *per parameter*, scaling down updates for weights with large gradients and scaling up for those with small gradients.

</details>

### Question 3 (Conceptual)
Why does Adam need bias correction in the first few steps?

<details><summary>💡 Answer</summary>

The moving averages $m$ and $v$ are initialized to zeros. In the early steps, they are biased heavily towards zero because the decay rates (e.g. 0.999) keep most of the previous zero value. Bias correction adjusts for this, ensuring the first steps have proper magnitude.

</details>